# en → ko | en-ko 데이터 증강 (back-translation)

변경 사항: **모델 구조는 베이스라인 그대로**(GRU 2층 + Luong attention), en-ko 데이터가 적어 `hid_dim=128`, `dropout=0.3`. 바꾼 것은 **학습 데이터**뿐.

- **원본**: 두 파일에 공통으로 있는 숫자 12,738쌍만 사용
- **증강**: 한쪽 파일에만 있는 숫자의 빈 언어를 num2en(김태우)·num2ko(홍정표) 모델로 생성해 en-ko 쌍 추가 → `make_loaders(..., extra_train=...)`
- **증강 데이터는 num2en·num2ko 모델이 생성, test는 원본만 사용** (valid/test 숫자는 생성 대상에서 제외)
- 번역은 en→ko Seq2Seq 하나가 직접 함. 숫자를 거치는 번역은 하지 않음 (다른 모델은 학습 데이터 생성에만 사용)
- num2en.pt가 Bahdanau라서 로드하려고 Decoder에 김태우의 `attention='bahdanau'` 분기를 포함 (CONFIG 옵션, 새 클래스 없음)
- 원본 모델과 증강 모델은 같은 `CONFIG`(GRU 2층 + Luong, hidden 128, dropout 0.3)를 사용해 비교하고, 명세서에 따라 test 정확도가 높은 쪽을 `en2ko.pt`로 저장
- **짧은 숫자 오버샘플링**: 증강 14만 쌍은 대부분 5\~10자리라 1\~3자리가 묻힘 (이전 실행에서 `zero` → 빈 출력, 3자리 97.3%). 원본 train의 1\~3자리 쌍을 `SHORT_REPEAT`번 반복해 `extra_train`에 추가 (모두 train split 숫자라 test 영향 없음)
- **beam 비교에 길이 패널티 beam 추가**: 베이스라인 `beam_decode`는 `len(finished) >= beam`이면 바로 종료해서 greedy보다 낮게 나옴. 홍정표(num2ko)의 `beam_decode_lp`를 가져와 같이 비교
- **공통 코드 수정 1곳**: `make_loaders`가 `extra_train`의 토큰도 vocab에 넣도록 함. 공통 숫자는 1\~8자리뿐이라 원본 vocab에는 `billion`, `억`이 없음 (`extra_train=None`이면 베이스라인과 동일)

In [14]:
# 설정
SRC, TGT = 'en', 'ko'  # num, en, ko

DATA_DIR = '../data'
SAVE_DIR = '..'  # pt 저장 위치 (num2en.pt, num2ko.pt도 여기서 읽음)

CONFIG = dict(
    emb_dim=128,
    hid_dim=128,     # en-ko는 데이터가 적어서 256 -> 128
    n_layers=2,
    dropout=0.3,     # 0.2 -> 0.3
    rnn='gru',  # gru / lstm
    attention='luong',  # 'luong' / 'bahdanau' / None
)
BATCH_SIZE = 128
EPOCHS_ORIG = 30  # 원본: train 약 1만 쌍이라 epoch를 늘림
EPOCHS_AUG = 10   # 증강: train이 훨씬 많아 epoch당 step 수가 큼
EPOCHS = EPOCHS_ORIG  # train() 기본값용
LR = 1e-3
TEACHER_FORCING = 0.5
BEAM_SIZE = 5
SEED = 42
MAX_TRAIN = None  # 디버깅용 train 샘플 수

SHORT_MAX_DIGITS = 3  # 이 자릿수 이하 원본 train 쌍을
SHORT_REPEAT = 20     # 이만큼 반복해서 증강 데이터에 추가 (0이면 오버샘플링 안 함)
LP_ALPHA = 0.6        # 길이 패널티 beam의 alpha

In [15]:
# Colab 실행 준비
# 이 노트북을 Colab에 직접 업로드해 실행하는 경우 사용.
%cd /content
!test -d /content/2026-nlp/.git || git clone https://github.com/jungyy808/2026-nlp.git
%cd /content/2026-nlp/project01/work

# num2en.pt, num2ko.pt는 project01/에서 읽음.
# 아직 main에 merge 전이면 Colab 왼쪽 폴더(/content)에 업로드해도 됨 (find_pt가 찾아줌)

/content
/content/2026-nlp/project01/work


In [16]:
import os, random, time, zlib, math
from collections import defaultdict

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

def set_seed(seed):
    random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

set_seed(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', DEVICE)

device: cuda


## 데이터 (베이스라인 그대로)

In [17]:
FILES = {'en': 'num2txt.txt', 'ko': 'num2kor.txt'}
PAD, SOS, EOS, UNK = 0, 1, 2, 3
SPECIALS = ['<pad>', '<sos>', '<eos>', '<unk>']


def tokenize(text, lang):
    # en: 단어 단위 / num, ko: 문자 단위 (공백 포함)
    return text.split() if lang == 'en' else list(text)


def detokenize(tokens, lang):
    return ' '.join(tokens) if lang == 'en' else ''.join(tokens)


def load_pairs(path):
    # CRLF 제거
    pairs = []
    with open(path, encoding='utf-8') as f:
        for line in f:
            line = line.rstrip('\r\n')
            if not line:
                continue
            num, text = line.split('\t', 1)
            pairs.append((num.strip(), text.strip()))
    return pairs


def get_pairs(src_lang, tgt_lang, data_dir=DATA_DIR):
    # (src, tgt, key) 반환, key는 split 기준
    assert src_lang != tgt_lang and {src_lang, tgt_lang} <= {'num', 'en', 'ko'}
    tables = {lang: dict(load_pairs(os.path.join(data_dir, FILES[lang])))
              for lang in (src_lang, tgt_lang) if lang != 'num'}
    if 'num' in (src_lang, tgt_lang):
        other = tgt_lang if src_lang == 'num' else src_lang
        keys = list(tables[other])
    else:  # en-ko: 공통 key만 사용
        keys = sorted(set(tables['en']) & set(tables['ko']))

    def text(lang, k):
        return k if lang == 'num' else tables[lang][k]

    return [(text(src_lang, k), text(tgt_lang, k), k) for k in keys]


def split_of(key):
    # key 해시 기준 8:1:1 분할
    b = zlib.crc32(key.encode('utf-8')) % 10
    return 'test' if b == 0 else 'valid' if b == 1 else 'train'


class Vocab:
    def __init__(self, lang, sentences=()):
        self.lang = lang
        tokens = sorted({t for s in sentences for t in tokenize(s, lang)})
        self.itos = SPECIALS + tokens
        self.stoi = {t: i for i, t in enumerate(self.itos)}

    @classmethod
    def from_itos(cls, lang, itos):
        v = cls(lang)
        v.itos = list(itos)
        v.stoi = {t: i for i, t in enumerate(v.itos)}
        return v

    def encode(self, text):  # sos/eos 제외
        return [self.stoi.get(t, UNK) for t in tokenize(text, self.lang)]

    def decode(self, ids):
        out = []
        for i in ids:
            i = int(i)
            if i == EOS:
                break
            if i in (PAD, SOS):
                continue
            out.append(self.itos[i])
        return detokenize(out, self.lang)

    def __len__(self):
        return len(self.itos)


class PairDataset(Dataset):
    def __init__(self, pairs, src_vocab, tgt_vocab):
        self.pairs = pairs
        self.src = [src_vocab.encode(s) for s, _, _ in pairs]
        self.tgt = [[SOS] + tgt_vocab.encode(t) + [EOS] for _, t, _ in pairs]

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, i):
        return self.src[i], self.tgt[i]


def collate(batch):
    srcs, tgts = zip(*batch)
    src_len = torch.tensor([len(s) for s in srcs])
    src = torch.full((len(srcs), max(src_len)), PAD, dtype=torch.long)
    tgt = torch.full((len(tgts), max(len(t) for t in tgts)), PAD, dtype=torch.long)
    for i, (s, t) in enumerate(zip(srcs, tgts)):
        src[i, :len(s)] = torch.tensor(s)
        tgt[i, :len(t)] = torch.tensor(t)
    return src, src_len, tgt


def make_loaders(pairs, src_lang, tgt_lang, batch_size=BATCH_SIZE, max_train=None, extra_train=None):
    # extra_train: 증강 등으로 추가할 (src, tgt, key) 리스트. train에만 들어가고 valid/test는 원본만 사용
    # [김수현 수정] extra_train의 토큰도 vocab에 포함 (extra_train=None이면 베이스라인과 동일)
    # en-ko 공통 숫자는 1~8자리뿐이라 원본만으로는 'billion', '억'이 vocab에 없음 -> 증강 데이터가 <unk>가 됨
    vocab_pairs = pairs + list(extra_train or [])
    src_vocab = Vocab(src_lang, [s for s, _, _ in vocab_pairs])
    tgt_vocab = Vocab(tgt_lang, [t for _, t, _ in vocab_pairs])
    parts = defaultdict(list)
    for p in pairs:
        parts[split_of(p[2])].append(p)
    if max_train:
        parts['train'] = random.Random(SEED).sample(parts['train'], min(max_train, len(parts['train'])))
    if extra_train:
        held_out = {p[2] for p in parts['valid'] + parts['test']}
        extra = [p for p in extra_train if p[2] not in held_out]  # valid/test 숫자와 겹치는 건 제외
        print(f'extra_train: {len(extra):,}개 추가 (valid/test와 겹쳐서 제외 {len(extra_train) - len(extra):,}개)')
        parts['train'] = parts['train'] + extra
    loaders = [DataLoader(PairDataset(parts[name], src_vocab, tgt_vocab), batch_size=batch_size,
                          shuffle=(name == 'train'), collate_fn=collate)
               for name in ('train', 'valid', 'test')]
    return (*loaders, src_vocab, tgt_vocab)

In [18]:
print('작업 위치:', os.getcwd())
for f in FILES.values():
    assert os.path.exists(os.path.join(DATA_DIR, f)), f'{os.path.abspath(os.path.join(DATA_DIR, f))} 없음 (Colab이면 clone 셀 확인)'
print('pt 저장 위치:', os.path.abspath(SAVE_DIR))

pairs = get_pairs(SRC, TGT)
train_loader, valid_loader, test_loader, src_vocab, tgt_vocab = make_loaders(
    pairs, SRC, TGT, BATCH_SIZE, MAX_TRAIN)

MAX_LEN = max(len(tokenize(t, TGT)) for _, t, _ in pairs) + 10  # 데이터 기준 최대 길이
print(f'{SRC} -> {TGT}: 전체 {len(pairs):,}쌍 | train {len(train_loader.dataset):,} / '
      f'valid {len(valid_loader.dataset):,} / test {len(test_loader.dataset):,}')
print(f'vocab: src {len(src_vocab)}, tgt {len(tgt_vocab)} | 디코딩 최대 길이 {MAX_LEN}')
for s, t, _ in pairs[:3]:
    print(f'  {s!r} -> {t!r}')

작업 위치: /content/2026-nlp/project01/work
pt 저장 위치: /content/2026-nlp/project01
en -> ko: 전체 12,738쌍 | train 10,162 / valid 1,274 / test 1,302
vocab: src 35, tgt 19 | 디코딩 최대 길이 26
  'zero' -> '영'
  'one' -> '일'
  'ten' -> '십'


## 모델 (베이스라인 + num2en.pt 로드용 Bahdanau 분기)

In [19]:
class Encoder(nn.Module):
    def __init__(self, vocab_size, emb_dim, hid_dim, n_layers, dropout, rnn):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, emb_dim, padding_idx=PAD)
        RNN = nn.LSTM if rnn == 'lstm' else nn.GRU
        self.rnn = RNN(emb_dim, hid_dim, n_layers, batch_first=True,
                       dropout=dropout if n_layers > 1 else 0)
        self.drop = nn.Dropout(dropout)

    def forward(self, src, src_len):
        x = self.drop(self.emb(src))
        packed = nn.utils.rnn.pack_padded_sequence(x, src_len.cpu(), batch_first=True, enforce_sorted=False)
        out, hidden = self.rnn(packed)
        out, _ = nn.utils.rnn.pad_packed_sequence(out, batch_first=True, total_length=src.size(1))
        return out, hidden


class Decoder(nn.Module):
    # 'bahdanau' 분기는 김태우(num2en) 구현을 그대로 가져옴 -> num2en.pt 로드에 필요
    def __init__(self, vocab_size, emb_dim, hid_dim, n_layers, dropout, rnn, attention):
        super().__init__()
        assert attention in ('luong', 'bahdanau', None), f'지원하지 않는 attention: {attention}'
        assert rnn in ('gru', 'lstm'), f'지원하지 않는 rnn: {rnn}'
        self.attention = attention
        self.emb = nn.Embedding(vocab_size, emb_dim, padding_idx=PAD)
        RNN = nn.LSTM if rnn == 'lstm' else nn.GRU
        rnn_in = emb_dim + hid_dim if attention == 'bahdanau' else emb_dim
        self.rnn = RNN(rnn_in, hid_dim, n_layers, batch_first=True,
                       dropout=dropout if n_layers > 1 else 0)
        if attention == 'luong':
            self.attn = nn.Linear(hid_dim, hid_dim, bias=False)  # Luong (general)
            self.combine = nn.Linear(hid_dim * 2, hid_dim)
        elif attention == 'bahdanau':
            self.W_h = nn.Linear(hid_dim, hid_dim, bias=False)
            self.W_s = nn.Linear(hid_dim, hid_dim, bias=False)
            self.v_a = nn.Linear(hid_dim, 1, bias=False)
            self.combine = nn.Linear(hid_dim * 2, hid_dim)
        self.out = nn.Linear(hid_dim, vocab_size)
        self.drop = nn.Dropout(dropout)

    def forward(self, prev_token, hidden, enc_out, src_mask):
        x = self.drop(self.emb(prev_token)).unsqueeze(1)
        if self.attention == 'bahdanau':
            query = hidden[0][-1] if isinstance(hidden, tuple) else hidden[-1]
            scores = self.v_a(torch.tanh(self.W_h(enc_out) + self.W_s(query).unsqueeze(1))).squeeze(-1)
            scores = scores.masked_fill(~src_mask, -1e9)
            ctx = torch.bmm(F.softmax(scores, dim=1).unsqueeze(1), enc_out).squeeze(1)
            h, hidden = self.rnn(torch.cat([x, ctx.unsqueeze(1)], dim=2), hidden)
            h = torch.tanh(self.combine(torch.cat([h.squeeze(1), ctx], dim=1)))
        else:
            h, hidden = self.rnn(x, hidden)
            h = h.squeeze(1)
            if self.attention == 'luong':
                scores = torch.bmm(enc_out, self.attn(h).unsqueeze(2)).squeeze(2)
                scores = scores.masked_fill(~src_mask, -1e9)
                ctx = torch.bmm(F.softmax(scores, dim=1).unsqueeze(1), enc_out).squeeze(1)
                h = torch.tanh(self.combine(torch.cat([h, ctx], dim=1)))
        return self.out(self.drop(h)), hidden


class Seq2Seq(nn.Module):
    def __init__(self, src_vocab_size, tgt_vocab_size, emb_dim=128, hid_dim=256,
                 n_layers=2, dropout=0.2, rnn='gru', attention='luong'):
        super().__init__()
        self.encoder = Encoder(src_vocab_size, emb_dim, hid_dim, n_layers, dropout, rnn)
        self.decoder = Decoder(tgt_vocab_size, emb_dim, hid_dim, n_layers, dropout, rnn, attention)

    def encode(self, src, src_len):
        enc_out, hidden = self.encoder(src, src_len)
        return (hidden, enc_out, src != PAD)  # (hidden, enc_out, mask)

    def decode_step(self, prev_token, state):
        hidden, enc_out, mask = state
        logits, hidden = self.decoder(prev_token, hidden, enc_out, mask)
        return logits, (hidden, enc_out, mask)

    def forward(self, src, src_len, tgt, teacher_forcing=0.5):
        # tgt[:, 1:] 예측
        state = self.encode(src, src_len)
        token, outputs = tgt[:, 0], []
        for t in range(1, tgt.size(1)):
            logits, state = self.decode_step(token, state)
            outputs.append(logits)
            use_tf = random.random() < teacher_forcing
            token = tgt[:, t] if use_tf else logits.argmax(1)
        return torch.stack(outputs, dim=1)


def reorder_state(state, idx):
    # beam search용 배치 재정렬
    hidden, enc_out, mask = state
    if isinstance(hidden, tuple):
        hidden = tuple(h.index_select(1, idx) for h in hidden)
    else:
        hidden = hidden.index_select(1, idx)
    return hidden, enc_out.index_select(0, idx), mask.index_select(0, idx)

## 학습, greedy / beam 디코딩, 평가, 저장 함수 (베이스라인 그대로)

In [20]:
@torch.no_grad()
def greedy_batch(model, src, src_len, max_len=None):
    # batch greedy
    max_len = max_len or getattr(model, 'max_len', MAX_LEN)  # 로드한 모델은 저장된 max_len 사용
    model.eval()
    state = model.encode(src, src_len)
    token = torch.full((src.size(0),), SOS, dtype=torch.long, device=src.device)
    done = torch.zeros(src.size(0), dtype=torch.bool, device=src.device)
    out = []
    for _ in range(max_len):
        logits, state = model.decode_step(token, state)
        token = logits.argmax(1)
        out.append(token)
        done |= token == EOS
        if done.all():
            break
    return torch.stack(out, dim=1)


def _to_tensor(text, vocab):
    ids = vocab.encode(text)
    return torch.tensor([ids], device=DEVICE), torch.tensor([len(ids)])


def greedy_decode(model, src_text, src_vocab, tgt_vocab, max_len=None):
    src, src_len = _to_tensor(src_text, src_vocab)
    return tgt_vocab.decode(greedy_batch(model, src, src_len, max_len)[0].tolist())


@torch.no_grad()
def beam_decode(model, src_text, src_vocab, tgt_vocab, beam=BEAM_SIZE, max_len=None):
    max_len = max_len or getattr(model, 'max_len', MAX_LEN)
    model.eval()
    src, src_len = _to_tensor(src_text, src_vocab)
    state = model.encode(src, src_len)
    beams = [([], 0.0)]  # (seq, log prob)
    finished = []
    for _ in range(max_len):
        tokens = torch.tensor([b[0][-1] if b[0] else SOS for b in beams], device=DEVICE)
        if len(beams) != state[1].size(0):
            state = reorder_state(state, torch.zeros(len(beams), dtype=torch.long, device=DEVICE))
        logits, state = model.decode_step(tokens, state)
        logp = F.log_softmax(logits, dim=1)
        cand = []
        for i, (seq, score) in enumerate(beams):
            top = logp[i].topk(beam)
            for lp, tok in zip(top.values.tolist(), top.indices.tolist()):
                cand.append((seq + [tok], score + lp, i))
        cand.sort(key=lambda c: c[1], reverse=True)
        beams, keep = [], []
        for seq, score, i in cand:
            if seq[-1] == EOS:
                finished.append((seq, score / len(seq)))  # 길이 정규화
            else:
                beams.append((seq, score)); keep.append(i)
            if len(beams) == beam:
                break
        if not beams or len(finished) >= beam:
            break
        state = reorder_state(state, torch.tensor(keep, device=DEVICE))
    if not finished:
        finished = [(seq, score / len(seq)) for seq, score in beams]
    best = max(finished, key=lambda f: f[1])[0]
    return tgt_vocab.decode(best)


@torch.no_grad()
def evaluate(model, loader, tgt_vocab):
    # exact match 기준
    model.eval()
    total_loss, n_tok, correct, n = 0.0, 0, 0, 0
    for src, src_len, tgt in loader:
        src, tgt = src.to(DEVICE), tgt.to(DEVICE)
        logits = model(src, src_len, tgt, teacher_forcing=1.0)
        gold = tgt[:, 1:]
        total_loss += F.cross_entropy(logits.reshape(-1, logits.size(-1)), gold.reshape(-1),
                                      ignore_index=PAD, reduction='sum').item()
        n_tok += (gold != PAD).sum().item()
        pred = greedy_batch(model, src, src_len)
        for p, g in zip(pred.tolist(), gold.tolist()):
            correct += tgt_vocab.decode(p) == tgt_vocab.decode(g)
            n += 1
    return {'loss': total_loss / n_tok, 'exact_match': correct / n}


def train(model, train_loader, valid_loader, epochs=EPOCHS, lr=LR, save_path=None):
    # best valid acc 모델 복원
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    history, best, best_state = [], -1, None
    for ep in range(1, epochs + 1):
        model.train()
        t0, total = time.time(), 0.0
        for src, src_len, tgt in train_loader:
            src, tgt = src.to(DEVICE), tgt.to(DEVICE)
            logits = model(src, src_len, tgt, TEACHER_FORCING)
            loss = F.cross_entropy(logits.reshape(-1, logits.size(-1)), tgt[:, 1:].reshape(-1), ignore_index=PAD)
            opt.zero_grad(); loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            total += loss.item()
        val = evaluate(model, valid_loader, tgt_vocab)
        history.append({'epoch': ep, 'train_loss': total / len(train_loader), **{f'valid_{k}': v for k, v in val.items()}})
        print(f"epoch {ep:2d} | train loss {history[-1]['train_loss']:.4f} | valid loss {val['loss']:.4f} "
              f"| valid acc {val['exact_match']*100:.2f}% | {time.time()-t0:.0f}s")
        if val['exact_match'] > best:
            best, best_state = val['exact_match'], {k: v.detach().clone() for k, v in model.state_dict().items()}
    model.load_state_dict(best_state)
    print(f'best valid acc: {best*100:.2f}%')
    return history

In [21]:
@torch.no_grad()
def predict_all(model, loader, tgt_vocab):
    rows, i = [], 0
    for src, src_len, _ in loader:
        pred = greedy_batch(model, src.to(DEVICE), src_len)
        for p in pred.tolist():
            s, t, k = loader.dataset.pairs[i]
            rows.append((s, t, tgt_vocab.decode(p), k)); i += 1
    return rows


def exact_match(rows):
    return sum(r[1] == r[2] for r in rows) / len(rows)


def acc_by_length(rows):
    stat = defaultdict(lambda: [0, 0])
    for _, gold, pred, key in rows:
        stat[len(key)][0] += gold == pred
        stat[len(key)][1] += 1
    print('자릿수 | 개수 | 정확도')
    for L in sorted(stat):
        c, n = stat[L]
        print(f'{L:6d} | {n:6d} | {c/n*100:6.2f}%')


def show_examples(rows, n=5):
    right = [r for r in rows if r[1] == r[2]][:n]
    wrong = [r for r in rows if r[1] != r[2]][:n]
    for title, group in (('맞은 예시', right), ('틀린 예시', wrong)):
        print(f'--- {title} ---')
        for s, gold, pred, _ in group:
            print(f'입력: {s}\n정답: {gold}\n예측: {pred}\n')



# 홍정표(num2ko) 구현: 길이 패널티 + 안전한 조기 종료 beam search
@torch.no_grad()
def beam_decode_lp(model, src_text, src_vocab, tgt_vocab, beam=BEAM_SIZE, alpha=LP_ALPHA, max_len=None):
    max_len = max_len or getattr(model, 'max_len', MAX_LEN)
    model.eval()
    src, src_len = _to_tensor(src_text, src_vocab)
    state = model.encode(src, src_len)
    beams, finished = [([], 0.0)], []

    def length_penalty(length):
        return ((5.0 + length) / 6.0) ** alpha

    for _ in range(max_len):
        tokens = torch.tensor([b[0][-1] if b[0] else SOS for b in beams], device=DEVICE)
        if len(beams) != state[1].size(0):
            state = reorder_state(state, torch.zeros(len(beams), dtype=torch.long, device=DEVICE))
        logits, state = model.decode_step(tokens, state)
        logp = F.log_softmax(logits, dim=1)
        cand = []
        for i, (seq, score) in enumerate(beams):
            top = logp[i].topk(beam)
            for lp_val, tok in zip(top.values.tolist(), top.indices.tolist()):
                cand.append((seq + [tok], score + lp_val, i))
        cand.sort(key=lambda c: c[1], reverse=True)
        beams, keep = [], []
        for seq, score, i in cand:
            if seq[-1] == EOS:
                finished.append((seq, score / length_penalty(len(seq))))
            else:
                beams.append((seq, score)); keep.append(i)
            if len(beams) == beam:
                break
        if not beams:
            break
        if finished:  # 남은 beam이 더 좋아질 수 없을 때만 종료
            if beams[0][1] / length_penalty(max_len) <= max(f[1] for f in finished):
                break
        state = reorder_state(state, torch.tensor(keep, device=DEVICE))
    if not finished:
        finished = [(seq, score / length_penalty(len(seq))) for seq, score in beams]
    return tgt_vocab.decode(max(finished, key=lambda f: f[1])[0])


def beam_accuracy(model, pairs, sv, tv, beam=BEAM_SIZE, decode=None):
    decode = decode or beam_decode
    return sum(decode(model, s, sv, tv, beam) == t for s, t, _ in pairs) / len(pairs)

In [22]:
def save_model(path, model, config, src_vocab, tgt_vocab, metrics=None):
    # 저장 형식 고정 (통합 시 로드)
    torch.save({
        'model': model.state_dict(),
        'config': config,
        'src_lang': src_vocab.lang, 'tgt_lang': tgt_vocab.lang,
        'src_vocab': src_vocab.itos, 'tgt_vocab': tgt_vocab.itos,
        'max_len': MAX_LEN,
        'metrics': metrics or {},
    }, path)
    print(f'saved: {path} ({os.path.getsize(path)/1e6:.1f} MB)')


def load_model(path):
    ckpt = torch.load(path, map_location=DEVICE)
    src_vocab = Vocab.from_itos(ckpt['src_lang'], ckpt['src_vocab'])
    tgt_vocab = Vocab.from_itos(ckpt['tgt_lang'], ckpt['tgt_vocab'])
    m = Seq2Seq(len(src_vocab), len(tgt_vocab), **ckpt['config']).to(DEVICE)
    m.load_state_dict(ckpt['model'])
    m.max_len = ckpt['max_len']
    m.eval()
    return m, src_vocab, tgt_vocab, ckpt


## 실험 1. 원본 데이터 (공통 숫자 12,738쌍)

In [23]:
set_seed(SEED)
orig_model = Seq2Seq(len(src_vocab), len(tgt_vocab), **CONFIG).to(DEVICE)
print('params:', sum(p.numel() for p in orig_model.parameters()))
print('train:', len(train_loader.dataset))
orig_history = train(orig_model, train_loader, valid_loader, EPOCHS_ORIG, LR)

params: 454931
train: 10162
epoch  1 | train loss 1.8038 | valid loss 0.6376 | valid acc 43.80% | 5s
epoch  2 | train loss 0.5870 | valid loss 0.1870 | valid acc 75.20% | 4s
epoch  3 | train loss 0.2096 | valid loss 0.0578 | valid acc 93.25% | 5s
epoch  4 | train loss 0.1097 | valid loss 0.0290 | valid acc 96.55% | 5s
epoch  5 | train loss 0.0613 | valid loss 0.0161 | valid acc 98.67% | 4s
epoch  6 | train loss 0.0480 | valid loss 0.0128 | valid acc 99.45% | 5s
epoch  7 | train loss 0.0303 | valid loss 0.0065 | valid acc 99.45% | 5s
epoch  8 | train loss 0.0231 | valid loss 0.0049 | valid acc 99.37% | 4s
epoch  9 | train loss 0.0175 | valid loss 0.0049 | valid acc 99.61% | 5s
epoch 10 | train loss 0.0174 | valid loss 0.0038 | valid acc 99.61% | 4s
epoch 11 | train loss 0.0104 | valid loss 0.0029 | valid acc 99.69% | 5s
epoch 12 | train loss 0.0109 | valid loss 0.0024 | valid acc 99.53% | 5s
epoch 13 | train loss 0.0176 | valid loss 0.0039 | valid acc 99.61% | 4s
epoch 14 | train loss 0

In [24]:
orig_src_vocab, orig_tgt_vocab, orig_test_loader = src_vocab, tgt_vocab, test_loader  # 원본 실험 vocab 보관
orig_rows = predict_all(orig_model, test_loader, tgt_vocab)
orig_acc = exact_match(orig_rows)
print(f'[원본] {SRC} -> {TGT} test exact match (greedy): {orig_acc*100:.2f}%  (n={len(orig_rows):,})\n')
acc_by_length(orig_rows)
print()
show_examples(orig_rows)

[원본] en -> ko test exact match (greedy): 99.77%  (n=1,302)

자릿수 | 개수 | 정확도
     1 |      1 | 100.00%
     2 |      8 | 100.00%
     3 |     74 | 100.00%
     4 |    913 | 100.00%
     5 |    277 |  99.64%
     6 |     24 |  95.83%
     7 |      4 | 100.00%
     8 |      1 |   0.00%

--- 맞은 예시 ---
입력: one thousand four
정답: 천사
예측: 천사

입력: one thousand ten
정답: 천십
예측: 천십

입력: one thousand eighteen
정답: 천십팔
예측: 천십팔

입력: one thousand nineteen
정답: 천십구
예측: 천십구

입력: ten thousand two hundred six
정답: 만 이백육
예측: 만 이백육

--- 틀린 예시 ---
입력: twenty nine million three hundred eleven thousand three hundred thirteen
정답: 이천구백삼십일만 천삼백십삼
예측: 이만 구백십삼

입력: thirty thousand five
정답: 삼만 오
예측: 삼만 오십

입력: eight hundred fifteen thousand two hundred fifty two
정답: 팔십일만 오천이백오십이
예측: 팔십만 오천이백오십이



## 실험 2. 데이터 증강 (back-translation)

| 경우 | 있는 것 | 만드는 것 | 생성 모델 |
|---|---|---|---|
| 영어 파일에만 있는 숫자 | 숫자, 영어 원문 | 한글 | num2ko.pt (홍정표) |
| 한글 파일에만 있는 숫자 | 숫자, 한글 원문 | 영어 | num2en.pt (김태우) |

생성한 쌍은 `extra_train`으로 train에만 들어감. valid/test는 원본 12,738쌍의 split 그대로라 원본 실험과 **같은 test로 비교**함.

In [13]:
# ============================================================
# en-ko 데이터 증강 (back-translation 방식)
#
# [무엇을 바꿨는가]
# - 두 파일 중 한쪽에만 있는 숫자는 en-ko 쌍을 만들 수 없어서 버려지고 있었음
# - 비어 있는 언어를 숫자 모델로 생성해서 en-ko 쌍으로 만듦
#     영어만 있는 숫자 -> num2ko.pt로 한글 생성 -> (영어 원문, 한글 생성)
#     한글만 있는 숫자 -> num2en.pt로 영어 생성 -> (영어 생성, 한글 원문)
#
# [왜 바꿨는가]
# - en-ko 공통 숫자는 12,738개뿐이라 train이 약 1만 쌍 (숫자 방향은 8만 쌍)
# - num2en / num2ko 모델은 test 정확도가 99.9%대라 생성 데이터 품질이 높음
#
# [지킨 규칙]
# - 번역은 en->ko Seq2Seq 하나가 직접 함. 숫자 모델은 학습 데이터 만들 때만 사용
# - valid/test에 속한 숫자는 생성 대상에서 아예 제외 (split_of(k) == 'train'만)
#   + make_loaders(extra_train=...)가 한 번 더 걸러줌 -> test는 원본만 사용
# ============================================================

@torch.no_grad()
def translate_batch(model, texts, sv, tv, batch_size=1024):
    # 여러 입력을 배치 greedy로 번역 (증강 데이터 생성용)
    outs = []
    for i in range(0, len(texts), batch_size):
        chunk = texts[i:i + batch_size]
        src, src_len, _ = collate([(sv.encode(t), [SOS]) for t in chunk])
        pred = greedy_batch(model, src.to(DEVICE), src_len)
        outs += [tv.decode(p) for p in pred.tolist()]
    return outs


def find_pt(name, save_dir=SAVE_DIR):
    # project01/ 에 없으면 Colab 업로드 위치(/content, 현재 폴더)도 확인
    for d in (save_dir, '.', '/content'):
        p = os.path.join(d, f'{name}.pt')
        if os.path.exists(p):
            return p
    raise FileNotFoundError(f'{name}.pt 없음: project01/에 merge됐는지 확인하거나 Colab 왼쪽 폴더에 업로드')


def build_augmented_pairs(data_dir=DATA_DIR, save_dir=SAVE_DIR):
    en = dict(load_pairs(os.path.join(data_dir, FILES['en'])))
    ko = dict(load_pairs(os.path.join(data_dir, FILES['ko'])))
    only_en = [k for k in en if k not in ko and split_of(k) == 'train']
    only_ko = [k for k in ko if k not in en and split_of(k) == 'train']

    gen_ko_model = load_model(find_pt('num2ko', save_dir))  # (model, sv, tv, ckpt)
    gen_en_model = load_model(find_pt('num2en', save_dir))
    assert gen_ko_model[1].lang == 'num' and gen_ko_model[2].lang == 'ko', 'num2ko.pt 언어 방향이 올바르지 않습니다.'
    assert gen_en_model[1].lang == 'num' and gen_en_model[2].lang == 'en', 'num2en.pt 언어 방향이 올바르지 않습니다.'
    t0 = time.time()
    gen_ko = translate_batch(gen_ko_model[0], only_en, *gen_ko_model[1:3])
    gen_en = translate_batch(gen_en_model[0], only_ko, *gen_en_model[1:3])
    print(f'생성 시간: {time.time() - t0:.0f}s')

    extra = ([(en[k], g, k) for k, g in zip(only_en, gen_ko)] +   # 영어 원문 + 한글 생성
             [(g, ko[k], k) for k, g in zip(only_ko, gen_en)])    # 영어 생성 + 한글 원문
    print(f'영어만 있는 train 숫자 {len(only_en):,}개 -> num2ko로 한글 생성')
    print(f'한글만 있는 train 숫자 {len(only_ko):,}개 -> num2en으로 영어 생성')
    print(f'증강 쌍 합계 {len(extra):,}개 (원본 train {len(train_loader.dataset):,}개)')
    return extra, gen_en_model, gen_ko_model


extra_pairs, gen_en_model, gen_ko_model = build_augmented_pairs()
print('\n증강 예시')
for s, t, k in extra_pairs[:2] + extra_pairs[-2:]:
    print(f'  [{k}] {s!r} -> {t!r}')

생성 시간: 9s
영어만 있는 train 숫자 69,866개 -> num2ko로 한글 생성
한글만 있는 train 숫자 70,159개 -> num2en으로 영어 생성
증강 쌍 합계 140,025개 (원본 train 10,162개)

증강 예시
  [3622185] 'three million six hundred twenty two thousand one hundred eighty five' -> '삼백육십이만 이천백팔십오'
  [49330605] 'forty nine million three hundred thirty thousand six hundred five' -> '사천구백삼십삼만 육백오'
  [88411795] 'eighty eight million four hundred eleven thousand seven hundred ninety five' -> '팔천팔백사십일만 천칠백구십오'
  [22878] 'twenty two thousand eight hundred seventy eight' -> '이만 이천팔백칠십팔'


In [25]:
# 생성 품질 확인: en-ko test 숫자(학습에 안 쓰는 숫자)로 생성 모델 정확도 측정
test_pairs = test_loader.dataset.pairs
keys = [k for _, _, k in test_pairs]
ko_gen = translate_batch(gen_ko_model[0], keys, *gen_ko_model[1:3])
en_gen = translate_batch(gen_en_model[0], keys, *gen_en_model[1:3])
print(f"num2ko 생성 정확도 (en-ko test 숫자 {len(keys):,}개): {sum(g == t for g, (_, t, _) in zip(ko_gen, test_pairs)) / len(keys) * 100:.2f}%")
print(f"num2en 생성 정확도 (en-ko test 숫자 {len(keys):,}개): {sum(g == s for g, (s, _, _) in zip(en_gen, test_pairs)) / len(keys) * 100:.2f}%")

# 원본 vocab 기준으로 생성 문장에 <unk>가 몇 개 생기는지 확인 -> make_loaders에서 extra_train 토큰도 vocab에 넣는 이유
print('원본 vocab에 없는 토큰:', sorted({w for s, _, _ in extra_pairs for w in s.split()} - set(src_vocab.itos)),
      sorted({c for _, t, _ in extra_pairs for c in t} - set(tgt_vocab.itos)))
unk_src = sum(UNK in src_vocab.encode(s) for s, _, _ in extra_pairs)
unk_tgt = sum(UNK in tgt_vocab.encode(t) for _, t, _ in extra_pairs)
print(f'증강 vocab 기준 <unk> 포함 쌍: src {unk_src:,}개, tgt {unk_tgt:,}개')
print('※ make_loaders는 extra_train 토큰까지 포함해 vocab을 만들기 때문에, 정상이라면 두 값 모두 0이어야 함.')

num2ko 생성 정확도 (en-ko test 숫자 1,302개): 100.00%
num2en 생성 정확도 (en-ko test 숫자 1,302개): 99.92%
원본 vocab에 없는 토큰: ['billion'] ['억']
증강 vocab 기준 <unk> 포함 쌍: src 24,140개, tgt 48,081개
※ make_loaders는 extra_train 토큰까지 포함해 vocab을 만들기 때문에, 정상이라면 두 값 모두 0이어야 함.


### 짧은 숫자 오버샘플링

증강 쌍은 대부분 5\~10자리라, 그대로 섞으면 1\~3자리 비중이 1% 미만으로 떨어짐. 이전 실행에서 증강 모델이 `zero`를 빈 문자열로, `one hundred twenty two`를 `백이십`으로 출력했던 원인. 원본 train의 짧은 숫자 쌍을 반복해서 넣어 비중을 맞춤. 모두 train split 숫자라 valid/test에는 영향 없음.

In [26]:
gen_pairs = extra_pairs  # 모델로 생성한 쌍만 (비교용으로 보관)
short_pairs = [p for p in train_loader.dataset.pairs if len(p[2]) <= SHORT_MAX_DIGITS]
extra_pairs = gen_pairs + short_pairs * SHORT_REPEAT

n_short = sum(len(p[2]) <= SHORT_MAX_DIGITS for p in train_loader.dataset.pairs + extra_pairs)
n_total = len(train_loader.dataset) + len(extra_pairs)
print(f'{SHORT_MAX_DIGITS}자리 이하 원본 train {len(short_pairs)}개 × {SHORT_REPEAT} 추가')
print(f'증강 train에서 {SHORT_MAX_DIGITS}자리 이하 비중: '
      f'{sum(len(p[2]) <= SHORT_MAX_DIGITS for p in train_loader.dataset.pairs + gen_pairs) / (len(train_loader.dataset) + len(gen_pairs)) * 100:.2f}% '
      f'-> {n_short / n_total * 100:.2f}%')
print('zero 포함 여부:', any(p[2] == '0' for p in short_pairs))

3자리 이하 원본 train 806개 × 20 추가
증강 train에서 3자리 이하 비중: 0.54% -> 10.18%
zero 포함 여부: True


In [27]:
aug_train_loader, aug_valid_loader, aug_test_loader, aug_src_vocab, aug_tgt_vocab = make_loaders(
    pairs, SRC, TGT, BATCH_SIZE, MAX_TRAIN, extra_train=extra_pairs)

# 원본 실험과 valid/test가 완전히 같은지 확인 (공정 비교)
assert [p[2] for p in aug_test_loader.dataset.pairs] == [p[2] for p in test_loader.dataset.pairs]
assert [p[2] for p in aug_valid_loader.dataset.pairs] == [p[2] for p in valid_loader.dataset.pairs]
test_keys = {p[2] for p in test_loader.dataset.pairs} | {p[2] for p in valid_loader.dataset.pairs}
assert not any(p[2] in test_keys for p in aug_train_loader.dataset.pairs)
print(f'train {len(train_loader.dataset):,} -> {len(aug_train_loader.dataset):,} | '
      f'valid {len(aug_valid_loader.dataset):,} / test {len(aug_test_loader.dataset):,} (원본과 동일)')
assert all(UNK not in ids for ids in aug_train_loader.dataset.src), '증강 source에 <unk>가 있습니다.'
assert all(UNK not in ids for ids in aug_train_loader.dataset.tgt), '증강 target에 <unk>가 있습니다.'

# 이후 증강 실험은 증강 vocab 사용 (train() 안의 evaluate가 전역 tgt_vocab을 씀)
src_vocab, tgt_vocab = aug_src_vocab, aug_tgt_vocab
MAX_LEN = max(len(tokenize(t, TGT)) for _, t, _ in pairs + extra_pairs) + 10
print(f'vocab: src {len(orig_src_vocab)} -> {len(src_vocab)}, tgt {len(orig_tgt_vocab)} -> {len(tgt_vocab)} | 디코딩 최대 길이 {MAX_LEN}')

extra_train: 156,145개 추가 (valid/test와 겹쳐서 제외 0개)
train 10,162 -> 166,307 | valid 1,274 / test 1,302 (원본과 동일)
vocab: src 35 -> 36, tgt 19 -> 20 | 디코딩 최대 길이 31


In [28]:
set_seed(SEED)
aug_model = Seq2Seq(len(src_vocab), len(tgt_vocab), **CONFIG).to(DEVICE)
aug_history = train(aug_model, aug_train_loader, aug_valid_loader, EPOCHS_AUG, LR)

epoch  1 | train loss 0.2093 | valid loss 0.0019 | valid acc 99.84% | 121s
epoch  2 | train loss 0.0102 | valid loss 0.0009 | valid acc 99.84% | 117s
epoch  3 | train loss 0.0062 | valid loss 0.0002 | valid acc 100.00% | 117s
epoch  4 | train loss 0.0052 | valid loss 0.0008 | valid acc 99.84% | 117s
epoch  5 | train loss 0.0035 | valid loss 0.0001 | valid acc 100.00% | 122s
epoch  6 | train loss 0.0024 | valid loss 0.0000 | valid acc 100.00% | 118s
epoch  7 | train loss 0.0032 | valid loss 0.0006 | valid acc 99.84% | 121s
epoch  8 | train loss 0.0039 | valid loss 0.0000 | valid acc 100.00% | 119s
epoch  9 | train loss 0.0030 | valid loss 0.0000 | valid acc 100.00% | 118s
epoch 10 | train loss 0.0024 | valid loss 0.0025 | valid acc 99.61% | 119s
best valid acc: 100.00%


In [29]:
aug_rows = predict_all(aug_model, aug_test_loader, tgt_vocab)
aug_acc = exact_match(aug_rows)
print(f'[증강] {SRC} -> {TGT} test exact match (greedy): {aug_acc*100:.2f}%  (n={len(aug_rows):,})\n')
acc_by_length(aug_rows)
print()
show_examples(aug_rows)

[증강] en -> ko test exact match (greedy): 100.00%  (n=1,302)

자릿수 | 개수 | 정확도
     1 |      1 | 100.00%
     2 |      8 | 100.00%
     3 |     74 | 100.00%
     4 |    913 | 100.00%
     5 |    277 | 100.00%
     6 |     24 | 100.00%
     7 |      4 | 100.00%
     8 |      1 | 100.00%

--- 맞은 예시 ---
입력: one thousand four
정답: 천사
예측: 천사

입력: one thousand ten
정답: 천십
예측: 천십

입력: one thousand eighteen
정답: 천십팔
예측: 천십팔

입력: one thousand nineteen
정답: 천십구
예측: 천십구

입력: ten thousand two hundred six
정답: 만 이백육
예측: 만 이백육

--- 틀린 예시 ---


## 결과 비교

In [30]:
orig_beam = beam_accuracy(orig_model, orig_test_loader.dataset.pairs, orig_src_vocab, orig_tgt_vocab)
aug_beam = beam_accuracy(aug_model, aug_test_loader.dataset.pairs, src_vocab, tgt_vocab)
orig_beam_lp = beam_accuracy(orig_model, orig_test_loader.dataset.pairs, orig_src_vocab, orig_tgt_vocab, decode=beam_decode_lp)
aug_beam_lp = beam_accuracy(aug_model, aug_test_loader.dataset.pairs, src_vocab, tgt_vocab, decode=beam_decode_lp)

print('=' * 98)
print(f"{'데이터':<10} {'train 쌍':>10} {'best epoch':>11} {'valid acc':>10} {'test greedy':>12} {'test beam5':>11} {'beam5+LP':>10}")
print('=' * 98)
for name, n_train, hist, g, b, blp in [
        ('원본', len(train_loader.dataset), orig_history, orig_acc, orig_beam, orig_beam_lp),
        ('증강', len(aug_train_loader.dataset), aug_history, aug_acc, aug_beam, aug_beam_lp)]:
    best = max(hist, key=lambda h: h['valid_exact_match'])
    print(f"{name:<10} {n_train:>10,} {best['epoch']:>11} {best['valid_exact_match']*100:>9.2f}% "
          f"{g*100:>11.2f}% {b*100:>10.2f}% {blp*100:>9.2f}%")
print(f'(beam5 = 베이스라인 beam_decode, beam5+LP = 길이 패널티 beam_decode_lp, alpha={LP_ALPHA})')

# 자릿수별 정확도 나란히
def by_len(rows):
    stat = defaultdict(lambda: [0, 0])
    for _, gold, pred, key in rows:
        stat[len(key)][0] += gold == pred; stat[len(key)][1] += 1
    return stat

so, sa = by_len(orig_rows), by_len(aug_rows)
print('\n자릿수 |  개수 |   원본  |   증강')
for L in sorted(so):
    print(f'{L:6d} | {so[L][1]:5d} | {so[L][0]/so[L][1]*100:6.2f}% | {sa[L][0]/sa[L][1]*100:6.2f}%')

o_map = {k: p for _, _, p, k in orig_rows}
improved = [(s, g, o_map[k], p) for s, g, p, k in aug_rows if o_map[k] != g and p == g]
regressed = [(s, g, o_map[k], p) for s, g, p, k in aug_rows if o_map[k] == g and p != g]
print(f'\n원본 오답 -> 증강 정답: {len(improved)}개 | 원본 정답 -> 증강 오답: {len(regressed)}개')
print('\n[증강으로 고쳐진 예시]')
for s, g, op, ap in improved[:5]:
    print(f'입력: {s}\n정답: {g}\n원본: {op}\n증강: {ap}\n')

데이터           train 쌍  best epoch  valid acc  test greedy  test beam5   beam5+LP
원본             10,162          27    100.00%       99.77%      92.55%     99.77%
증강            166,307           3    100.00%      100.00%      51.15%    100.00%
(beam5 = 베이스라인 beam_decode, beam5+LP = 길이 패널티 beam_decode_lp, alpha=0.6)

자릿수 |  개수 |   원본  |   증강
     1 |     1 | 100.00% | 100.00%
     2 |     8 | 100.00% | 100.00%
     3 |    74 | 100.00% | 100.00%
     4 |   913 | 100.00% | 100.00%
     5 |   277 |  99.64% | 100.00%
     6 |    24 |  95.83% | 100.00%
     7 |     4 | 100.00% | 100.00%
     8 |     1 |   0.00% | 100.00%

원본 오답 -> 증강 정답: 3개 | 원본 정답 -> 증강 오답: 0개

[증강으로 고쳐진 예시]
입력: twenty nine million three hundred eleven thousand three hundred thirteen
정답: 이천구백삼십일만 천삼백십삼
원본: 이만 구백십삼
증강: 이천구백삼십일만 천삼백십삼

입력: thirty thousand five
정답: 삼만 오
원본: 삼만 오십
증강: 삼만 오

입력: eight hundred fifteen thousand two hundred fifty two
정답: 팔십일만 오천이백오십이
원본: 팔십만 오천이백오십이
증강: 팔십일만 오천이백오십이



## 참고 평가: 9\~10자리 포함 긴 숫자

공식 지표는 위의 원본 test(공통 숫자, 1\~8자리)지만, 이 test에는 9\~10자리가 없음. 최종 함수 `en2ko()`는 10자리까지 받아야 하므로 참고용으로 따로 확인함.
- 대상: 한글 파일에만 있는 **test split** 숫자 (학습·증강에 안 쓴 숫자)
- 정답: 원본 한글 / 입력 영어: num2en.pt로 생성 (생성 정확도 99.9%대)
- 자릿수별 최대 200개 샘플. 공식 test 정확도가 아니라 참고 수치

In [31]:
en_all = dict(load_pairs(os.path.join(DATA_DIR, FILES['en'])))
ko_all = dict(load_pairs(os.path.join(DATA_DIR, FILES['ko'])))
by_digit = defaultdict(list)
for k in ko_all:
    if k not in en_all and split_of(k) == 'test':
        by_digit[len(k)].append(k)
rng = random.Random(SEED)
long_keys = [k for L in sorted(by_digit) for k in rng.sample(by_digit[L], min(200, len(by_digit[L])))]
long_src = translate_batch(gen_en_model[0], long_keys, *gen_en_model[1:3])
long_pairs = [(s, ko_all[k], k) for s, k in zip(long_src, long_keys)]

def rows_for(model, sv, tv, pairs_):
    preds = translate_batch(model, [s for s, _, _ in pairs_], sv, tv)
    return [(s, t, p, k) for (s, t, k), p in zip(pairs_, preds)]

lo = by_len(rows_for(orig_model, orig_src_vocab, orig_tgt_vocab, long_pairs))
la = by_len(rows_for(aug_model, src_vocab, tgt_vocab, long_pairs))
print('[참고] 긴 숫자 평가 (입력은 num2en 생성, 정답은 원본 한글)')
print('자릿수 |  개수 |   원본  |   증강')
for L in sorted(lo):
    print(f'{L:6d} | {lo[L][1]:5d} | {lo[L][0]/lo[L][1]*100:6.2f}% | {la[L][0]/la[L][1]*100:6.2f}%')
tot = lambda st: sum(v[0] for v in st.values()) / sum(v[1] for v in st.values())
print(f'전체   | {sum(v[1] for v in lo.values()):5d} | {tot(lo)*100:6.2f}% | {tot(la)*100:6.2f}%')


[참고] 긴 숫자 평가 (입력은 num2en 생성, 정답은 원본 한글)
자릿수 |  개수 |   원본  |   증강
     5 |   200 |  99.50% |  99.50%
     6 |   200 |  97.50% | 100.00%
     7 |   200 |  80.50% |  99.00%
     8 |   200 |   0.00% |  99.50%
     9 |   200 |   0.00% | 100.00%
    10 |   200 |   0.00% | 100.00%
전체   |  1200 |  46.25% |  99.67%


## 저장 (test 정확도가 높은 쪽을 en2ko.pt로)

In [32]:
use_aug = aug_acc >= orig_acc
best_model, best_history, best_acc = (aug_model, aug_history, aug_acc) if use_aug else (orig_model, orig_history, orig_acc)
print('선택:', '증강 모델' if use_aug else '원본 모델', f'(test {best_acc*100:.2f}%)')

SAVE_PATH = os.path.join(SAVE_DIR, f'{SRC}2{TGT}.pt')
best_sv, best_tv = (src_vocab, tgt_vocab) if use_aug else (orig_src_vocab, orig_tgt_vocab)
save_model(SAVE_PATH, best_model, CONFIG, best_sv, best_tv,
           metrics={'test_exact_match': best_acc, 'history': best_history,
                    'augmented': use_aug,
                    'compare': {'orig_test': orig_acc, 'aug_test': aug_acc,
                                'orig_beam5': orig_beam, 'aug_beam5': aug_beam,
                                'orig_beam5_lp': orig_beam_lp, 'aug_beam5_lp': aug_beam_lp,
                                'short_oversample': {'max_digits': SHORT_MAX_DIGITS, 'repeat': SHORT_REPEAT},
                                'orig_train': len(train_loader.dataset),
                                'aug_train': len(aug_train_loader.dataset),
                                'long_orig': tot(lo), 'long_aug': tot(la)}})

# Colab이면 바로 다운로드 (런타임 끊기면 파일이 사라짐)
try:
    from google.colab import files
    files.download(SAVE_PATH)
except ImportError:
    pass

선택: 증강 모델 (test 100.00%)
saved: ../en2ko.pt (1.8 MB)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 불러와서 테스트 / 사용법

```python
m, sv, tv, ckpt = load_model('../en2ko.pt')
greedy_decode(m, 'twelve thousand three hundred forty five', sv, tv)   # '만 이천삼백사십오'
beam_decode_lp(m, 'twelve thousand three hundred forty five', sv, tv, beam=5)  # 베이스라인 beam_decode보다 안정적
```
입력은 파일 표기(소문자, 복수형 없음: `thousand`, `hundred`). 슬라이드 표기(`thousands`)는 통합 노트북에서 이재노의 정규화 함수로 처리.

과제 슬라이드 표기(`thousands`, `hundreds` 복수형)도 `en2ko()`가 단수로 바꿔서 처리함. 출력은 데이터 파일(num2kor.txt) 표기를 따름 (예: `만 이천삼백사십오`).

In [33]:
# ============================================================
# 사용 방법
#   m, sv, tv, ckpt = load_model('../en2ko.pt')     # project01/en2ko.pt
#   en2ko('twelve thousand three hundred forty five')          # greedy -> '만 이천삼백사십오'
#   en2ko('twelve thousands three hundreds forty five', 'beam') # 슬라이드 표기(복수형)도 가능
# 입력은 영어 문자열 (숫자를 int로 다루지 않음), 출력은 num2kor.txt 표기를 따름
# ============================================================
m, sv, tv, ckpt = load_model(SAVE_PATH)
print(f"loaded {ckpt['src_lang']} -> {ckpt['tgt_lang']} | augmented={ckpt['metrics']['augmented']} "
      f"| test acc {ckpt['metrics']['test_exact_match']*100:.2f}%\n")

# 과제 슬라이드는 'thousands', 'hundreds' 같은 복수형을 쓰지만 데이터 파일은 단수형이라
# 복수형 단위어만 단수로 바꿔서 모델에 넣음 (번역 자체는 Seq2Seq가 함)
PLURAL = {'hundreds': 'hundred', 'thousands': 'thousand', 'millions': 'million', 'billions': 'billion'}

def en2ko(text, decoding='greedy'):
    words = [PLURAL.get(w, w) for w in text.strip().lower().replace(',', ' ').replace('-', ' ').split()]
    text = ' '.join(words)
    if decoding == 'beam':
        return beam_decode_lp(m, text, sv, tv, beam=BEAM_SIZE)
    return greedy_decode(m, text, sv, tv)

checks = [('zero', '영'), ('seven', '칠'), ('one hundred twenty two', '백이십이'),
          ('five hundred twenty one', '오백이십일'),
          ('twelve thousand three hundred forty five', '만 이천삼백사십오'),
          ('twelve thousands three hundreds forty five', '만 이천삼백사십오'),   # 슬라이드 예시 표기
          ('four thousand five hundred seven', '사천오백칠'),
          ('one billion two hundred thirty four million five hundred sixty seven thousand eight hundred ninety',
           '십이억 삼천사백오십육만 칠천팔백구십')]
for s, gold in checks:
    g, b = en2ko(s), en2ko(s, 'beam')
    mark = 'O' if g == gold and b == gold else 'X'
    print(f'[{mark}] {s!r}\n    greedy: {g} | beam: {b} | 정답: {gold}')

print()
for s, t, _ in test_loader.dataset.pairs[:3]:   # 원본 test 예시
    print(f'입력 {s!r}\n  예측: {en2ko(s)}\n  정답: {t}')

loaded en -> ko | augmented=True | test acc 100.00%

[O] 'zero'
    greedy: 영 | beam: 영 | 정답: 영
[O] 'seven'
    greedy: 칠 | beam: 칠 | 정답: 칠
[O] 'one hundred twenty two'
    greedy: 백이십이 | beam: 백이십이 | 정답: 백이십이
[O] 'five hundred twenty one'
    greedy: 오백이십일 | beam: 오백이십일 | 정답: 오백이십일
[O] 'twelve thousand three hundred forty five'
    greedy: 만 이천삼백사십오 | beam: 만 이천삼백사십오 | 정답: 만 이천삼백사십오
[O] 'twelve thousands three hundreds forty five'
    greedy: 만 이천삼백사십오 | beam: 만 이천삼백사십오 | 정답: 만 이천삼백사십오
[O] 'four thousand five hundred seven'
    greedy: 사천오백칠 | beam: 사천오백칠 | 정답: 사천오백칠
[O] 'one billion two hundred thirty four million five hundred sixty seven thousand eight hundred ninety'
    greedy: 십이억 삼천사백오십육만 칠천팔백구십 | beam: 십이억 삼천사백오십육만 칠천팔백구십 | 정답: 십이억 삼천사백오십육만 칠천팔백구십

입력 'one thousand four'
  예측: 천사
  정답: 천사
입력 'one thousand ten'
  예측: 천십
  정답: 천십
입력 'one thousand eighteen'
  예측: 천십팔
  정답: 천십팔
